# Phase N — iNTERPRET
## Portfolio Construction → Backtest → Performance Tearsheet

**OSEMN Step 5**: Apply regime probabilities to build the portfolio and evaluate performance.

Key steps:
1. Probability-weighted ETF allocation (smooth blending, no binary switching)
2. Threshold rebalancing with realistic transaction costs (7bps round-trip)
3. Walk-forward backtest vs 5 benchmarks
4. Full performance tear sheet (Sharpe, Calmar, CVaR, etc.)
5. Crisis period analysis (6 events)
6. Visual outputs: equity curves, allocation chart, transition heatmap

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

from src.config_loader import load_config
from src.obtain.fred_loader import FREDLoader
from src.obtain.market_loader import MarketLoader
from src.scrub.preprocessor import Preprocessor
from src.scrub.feature_engineer import FeatureEngineer
from src.model.hmm_trainer import HMMTrainer
from src.portfolio.allocator import MacroAllocator
from src.portfolio.rebalancer import Rebalancer
from src.interpret.backtest import Backtester
from src.interpret.metrics import PerformanceMetrics
from src.interpret.visualize import Visualizer
from src.explore.eda_plots import EDAPlots

cfg = load_config('../config/config.yaml')

# ── Load all data ───────────────────────────────────────────────────────
fred_data = FREDLoader(cfg).load()
market_data = MarketLoader(cfg).load()
prep = Preprocessor(cfg)
df_zscored, df_raw = prep.build_feature_matrix(fred_data, save=False)
features = FeatureEngineer(cfg).engineer(df_raw, df_zscored, save=False)
assets = list(cfg['data']['market']['etfs'].keys())
returns = prep.build_returns_matrix(market_data, assets, save=False)
print('All data loaded.')

## 5.1 Load Saved HMM Model
Loads the model saved in notebook 04. Run 04 first.

In [ ]:
BEST_K = 3   # ← update from 03_explore.ipynb BIC result
OOS_START = cfg['exploration']['oos_start']

trainer = HMMTrainer(cfg)

# Re-run walk-forward (or load saved regime_probs if you cached it separately)
regime_probs = trainer.walk_forward(features.dropna(), BEST_K, OOS_START)
full_model = trainer.load('hmm_full')
feature_names = features.columns.tolist()
regime_names = trainer.label_regimes(full_model, feature_names, BEST_K)
print(f'Regime names: {regime_names}')

# Rename columns
for k, name in enumerate(regime_names):
    if f'prob_{k}' in regime_probs.columns:
        regime_probs.rename(columns={f'prob_{k}': f'prob_{name}'}, inplace=True)
print(regime_probs.tail())

## 5.2 Compute Portfolio Weights

In [ ]:
allocator = MacroAllocator(cfg)

# Build generic prob_K columns for allocator
probs = regime_probs.copy()
for k, name in enumerate(regime_names):
    if f'prob_{name}' in probs.columns:
        probs[f'prob_{k}'] = probs[f'prob_{name}']

target_weights = allocator.compute_weights(probs, regime_names)
print(f'Target weights shape: {target_weights.shape}')
print(f'\nMean allocation:')
print((target_weights.mean() * 100).round(1).to_string())

## 5.3 Visualise Allocation Over Time

In [ ]:
viz = Visualizer(cfg)
viz.allocation_area(target_weights)

# Quick in-notebook preview
import matplotlib.pyplot as plt
target_weights.plot.area(figsize=(16, 6), alpha=0.8, colormap='tab10')
plt.title('Portfolio Allocation Over Time')
plt.ylabel('Weight')
plt.legend(loc='upper left', ncol=3)
plt.tight_layout()
plt.show()

## 5.4 Simulate with Transaction Costs

In [ ]:
rebalancer = Rebalancer(cfg)
sim = rebalancer.simulate(target_weights, returns)

print(f'Simulation: {len(sim)} months')
print(f'Total rebalances:     {sim["rebalanced"].sum()}')
print(f'Avg monthly turnover: {sim["turnover"].mean():.1%}')
print(f'Total TC drag (ann.): {sim["tc_drag"].mean()*12*100:.2f} bps')

strategy_returns = sim['portfolio_return']

## 5.5 Backtest vs Benchmarks

In [ ]:
inv_vol_weights = allocator.inverse_vol_static(returns)
backtester = Backtester(cfg)
all_results = backtester.run(strategy_returns, returns, inv_vol_weights)

print('Equity curve shapes:')
for name, df in all_results.items():
    final = df['cumulative_return'].iloc[-1]
    print(f'  {name:30s}: {final:.2f}x total return')

## 5.6 Full Performance Tear Sheet

In [ ]:
metrics_engine = PerformanceMetrics(cfg)
numeric_df, display_df = metrics_engine.comparison_table(all_results)

print('=== PERFORMANCE COMPARISON ===')
display_df

## 5.7 Crisis Period Analysis

In [ ]:
crisis_df = backtester.crisis_analysis(all_results)
print('=== CRISIS PERIOD ANALYSIS ===')
crisis_df.pivot(index='crisis', columns='strategy', values='max_drawdown')

## 5.8 Visualisations

In [ ]:
# Equity curves + drawdown
viz.equity_and_drawdown(all_results)

# Transition heatmap
viz.transition_heatmap(full_model.transmat_, regime_names)

# Performance bars
viz.performance_bars(numeric_df)

# Rolling Sharpe
viz.rolling_sharpe(all_results)

# Regime timeline
spy_prices = market_data['SPY']['Close'].resample(
    cfg['data']['preprocessing']['resample_freq']).last()
EDAPlots(cfg).regime_timeline(probs, spy_prices, regime_names)

print('All figures saved to outputs/figures/')

## 5.9 Quantstats Extended Tear Sheet (optional)

In [ ]:
try:
    import quantstats as qs
    spy_daily = market_data['SPY']['Close'].pct_change().dropna()
    report_path = '../outputs/reports/quantstats_tearsheet.html'
    qs.reports.html(
        strategy_returns,
        benchmark=all_results['Buy & Hold SPY']['return'],
        output=report_path,
        title='Macro Regime Strategy'
    )
    print(f'QuantStats report saved → {report_path}')
except ImportError:
    print('quantstats not installed. Run: pip install quantstats')
except Exception as e:
    print(f'QuantStats failed: {e}')

## Summary
- Probability-weighted portfolio weights computed from HMM output
- Monthly threshold rebalancing (2% bands) with 7bps round-trip TC
- Walk-forward OOS equity curve vs 4+ benchmarks
- Full performance tear sheet (Sharpe, Calmar, Sortino, CVaR, Omega)
- Crisis-period drawdown comparison
- All charts saved to `outputs/figures/`

**Project Complete** — next steps:
- Tune `config.yaml` based on EDA findings
- Add NVDA/WDC stock overlay (adaptive position sizing formula)
- Add FinBERT sentiment signal
- Write GitHub README with results summary